# 01 – Ingest playground

Use this notebook to experiment with document ingestion into ChromaDB.

**Before running**, make sure:
1. You have a `.env` file with `OPENAI_API_KEY=sk-...`
2. Dependencies are installed: `pip install -r requirements.txt`
3. At least one document is present under `data/raw/`

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # repo root on PYTHONPATH

from rag import config
print('RAW data dir:', config.DATA_RAW)
print('Chroma dir:  ', config.CHROMA_PERSIST_DIR)
print('Embedding model:', config.EMBEDDING_MODEL)

In [ ]:
# List all files that will be ingested
supported = {'.pdf', '.html', '.htm', '.txt', '.md'}
files = [p for p in config.DATA_RAW.rglob('*') if p.is_file() and p.suffix.lower() in supported]
for f in files:
    print(f)

In [ ]:
# Preview chunking on a single file
from rag.ingest import _extract_text, _chunk_text

# Change this path to an actual file in data/raw/
sample_path = next(iter(files), None)
if sample_path:
    text = _extract_text(sample_path)
    chunks = _chunk_text(text)
    print(f'{len(chunks)} chunks from {sample_path.name}')
    print('--- first chunk ---')
    print(chunks[0][:500])
else:
    print('No files found – drop a PDF or .txt into data/raw/ first')

In [ ]:
# Run a full ingest (costs API credits for embeddings)
from rag.ingest import ingest_all

total = ingest_all()
print(f'Ingested {total} chunks total')

In [ ]:
# Inspect the collection
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

client = chromadb.PersistentClient(path=str(config.CHROMA_PERSIST_DIR))
ef = OpenAIEmbeddingFunction(api_key=config.OPENAI_API_KEY, model_name=config.EMBEDDING_MODEL)
col = client.get_or_create_collection(config.CHROMA_COLLECTION, embedding_function=ef)
print('Collection count:', col.count())
sample = col.peek(5)
for doc, meta in zip(sample['documents'], sample['metadatas']):
    print(meta['filename'], '|', doc[:120])